# Topic 7 — Linear Regression
### Theory → tiny example → from-scratch implementation → sklearn → experiment.

Linear regression predicts a **continuous number** from one or more input features by fitting a
straight line (or hyperplane) through the data.

```text
y = wx + b            (one feature — simple linear regression)
y = Xw + b             (many features — multiple linear regression, matrix form)
```

`w` (weight/slope) and `b` (bias/intercept) are the model's **parameters**, learned by minimizing a
**loss function** via **gradient descent** (both covered in Topic 4/5 — this is where they pay off).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error

rng = np.random.default_rng(0)

## 1. Simple linear regression — one feature

`y = wx + b`. We'll generate data from a known line plus noise, so we can check if the model
recovers something close to the true w and b.

In [ ]:
true_w, true_b = 3.0, 5.0
X = rng.uniform(0, 10, 50).reshape(-1, 1)
y = true_w * X.flatten() + true_b + rng.normal(0, 2, 50)   # add noise

plt.figure(figsize=(5, 4))
plt.scatter(X, y, alpha=0.7)
plt.title("Data generated from y = 3x + 5 + noise")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

## 2. Residuals

A **residual** is the difference between the actual value and the predicted value: `residual = y - y_pred`.
Residuals are the raw material every regression loss function is built from.

In [ ]:
model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
residuals = y - y_pred

print("learned w:", model.coef_[0], " (true w=3)")
print("learned b:", model.intercept_, " (true b=5)")

plt.figure(figsize=(5, 4))
plt.scatter(X, residuals)
plt.axhline(0, color="red", linestyle="--")
plt.title("Residuals (should scatter randomly around 0)")
plt.xlabel("x"); plt.ylabel("residual")
plt.show()
# If residuals show a clear PATTERN (e.g. a curve), a straight line is the wrong model for this data.

## 3. Loss functions: MSE, MAE, RMSE

- **MSE** (Mean Squared Error): average of squared residuals. Penalizes big errors heavily.
- **MAE** (Mean Absolute Error): average of absolute residuals. Treats all errors proportionally.
- **RMSE**: square root of MSE — brings the error back into the same units as `y`, easier to interpret.

In [ ]:
mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mse)

print(f"MSE:  {mse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}  <- roughly 'typical prediction error in the same units as y'")

## 4. From-scratch implementation with gradient descent

This is the important exercise from the roadmap: implement linear regression yourself,
using the gradient descent loop from Topic 4/5, so `w` and `b` are learned, not solved analytically.

In [ ]:
def train_linear_regression(X, y, lr=0.01, epochs=1000):
    n = len(y)
    w, b = 0.0, 0.0
    X_flat = X.flatten()
    loss_history = []

    for epoch in range(epochs):
        y_pred = w * X_flat + b
        loss = np.mean((y - y_pred) ** 2)           # MSE loss
        loss_history.append(loss)

        # gradients of MSE loss w.r.t. w and b (calculus from Topic 4)
        dw = (-2 / n) * np.sum(X_flat * (y - y_pred))
        db = (-2 / n) * np.sum(y - y_pred)

        w -= lr * dw          # gradient descent update
        b -= lr * db

    return w, b, loss_history

# Normalize X first -- gradient descent converges much better on similar-scale features
X_norm = (X.flatten() - X.mean()) / X.std()
w, b, loss_history = train_linear_regression(X_norm.reshape(-1, 1), y, lr=0.1, epochs=500)

print("scratch-learned w, b (on normalized X):", w, b)

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoch"); plt.ylabel("MSE loss")
plt.title("Loss decreasing during training")
plt.show()
# --- Try it yourself: set lr=0.5 or lr=0.001 and see how the loss curve changes.

## 5. Multiple linear regression — several features

`y = Xw + b`, where X now has multiple columns. Matrix form scales to any number of features.

In [ ]:
# 2 features: hours_studied, hours_slept -> predict exam_score
n_samples = 100
hours_studied = rng.uniform(0, 10, n_samples)
hours_slept = rng.uniform(4, 9, n_samples)

true_w = np.array([5.0, 2.0])
true_b = 20.0
exam_score = true_w[0]*hours_studied + true_w[1]*hours_slept + true_b + rng.normal(0, 5, n_samples)

X_multi = np.column_stack([hours_studied, hours_slept])   # shape (100, 2)

multi_model = LinearRegression()
multi_model.fit(X_multi, exam_score)

print("learned weights:", multi_model.coef_, " (true:", true_w, ")")
print("learned bias:", multi_model.intercept_, " (true:", true_b, ")")

# Interpretation: each weight tells you how much the prediction changes per 1-unit
# increase in that feature, holding the other features constant.

## 6. Polynomial regression

Still "linear" regression in the mathematical sense (linear in the parameters), but you first
transform features into polynomial terms (`x, x², x³...`) so it can fit curves, not just straight lines.

In [ ]:
X_curve = np.linspace(0, 10, 40).reshape(-1, 1)
y_curve = 0.5 * X_curve.flatten()**2 - 3*X_curve.flatten() + 10 + rng.normal(0, 3, 40)

linear_fit = LinearRegression().fit(X_curve, y_curve)
poly_fit = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(X_curve, y_curve)

x_line = np.linspace(0, 10, 200).reshape(-1, 1)

plt.figure(figsize=(6, 4))
plt.scatter(X_curve, y_curve, alpha=0.6, label="data")
plt.plot(x_line, linear_fit.predict(x_line), label="degree=1 (straight line)", color="orange")
plt.plot(x_line, poly_fit.predict(x_line), label="degree=2 (curve)", color="green")
plt.legend()
plt.title("Polynomial regression captures curved relationships")
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Generate your own noisy data from y = -2x + 10 + noise, fit sklearn's LinearRegression,
#    and check how close the learned w,b are to -2 and 10.
# 2. Modify train_linear_regression() above to also track and print RMSE (not just MSE) each 100 epochs.
# 3. Build a multiple regression with 3 features (make one of them irrelevant/random noise) and see
#    whether its learned weight comes out close to 0.
# 4. Fit polynomial degree=1, 2, and 8 to the curved data above and compare test-set behavior
#    (this connects directly back to the underfit/overfit demo in Topic 5).

---
### Next up: **Topic 8 — Logistic Regression** (your first classifier — very important for text classification).

Say "next" when you're ready.